# Tabela de atividades de usuários: users_usage
Criação da tabela de atividades por usuário.
- **email** é o campo de chave estrangeira
- **tabela** não possui chaves primárias

# Pipeline: users_usage
A **pipeline** de dados da tabela users_usage consistente da inserção de registros da tabela sistemica **system.access.audit**, sendo essa a tabela sistemica que armazena as informações de todas as ações realizadas no ambiente Databricks.
Além disso existe a comunicação com a tabela de usuários (users) que serve para armazenar os usuários distintos do ambiente.
- Exclui a tabela, se ela existir;
- Realiza a criação da tabela;
- Atribui ao campo "email" a caracteristica de chave estrangeira;
- Realiza o streaming de dados com a origem da informação;
- Realiza a insersão dos valores na tabela gerada.

In [0]:
%sql
DROP TABLE IF EXISTS data_governance.users_usage;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS data_governance.users_usage(
  email STRING NOT NULL
  , usage_log STRUCT <
    service_name: STRING
    , action_name: STRING
    >
    ,event_date DATE);

In [0]:
%sql
ALTER TABLE data_governance.users_usage ADD CONSTRAINT fk_email_usage FOREIGN KEY(email) REFERENCES data_governance.users(email);

In [0]:
%sql
COMMENT ON TABLE data_governance.users_usage IS '## Content
This table has information about each user usage of the workspace.
### Update
This table updates every day 7:00 AM (UTC-03:00)';
COMMENT ON COLUMN data_governance.users_usage.email IS 'Email of each user, it is the primary key of the table';
COMMENT ON COLUMN data_governance.users_usage.usage_log IS '# Struct with the service name and action name
- service_name:The name of the service that is was used by user;
- action_name: The action that was performed by the user;';
COMMENT ON COLUMN data_governance.users_usage.event_date IS 'The date when the action was performed';

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE data_governance.users_usage(
  email STRING
  , usage_log STRUCT <
    service_name: STRING
    , action_name: STRING
    >,
    event_date DATE
) AS SELECT
    u.email
    , s.service_name
    , s.action_name
    , s.event_date
  FROM data_governance.users u
  LEFT JOIN STREAM system.access.audit s ON u.email = s.user_identity.email
  

In [0]:
%sql
INSERT INTO data_governance.users_usage(
  email,
  usage_log,
  event_date
)
SELECT
  u.email,
  named_struct('service_name', s.service_name, 'action_name', s.action_name) AS usage_log,
  s.event_date
FROM data_governance.users u
LEFT JOIN system.access.audit s
  ON  u.email = s.user_identity.email
WHERE NOT EXISTS (
  SELECT 1
  FROM data_governance.users_usage ac
  WHERE ac.email = u.email
    AND ac.usage_log.service_name = s.service_name
    AND ac.usage_log.action_name = s.action_name
    AND ac.event_date = s.event_date
)

In [0]:
%sql
SELECT * FROM data_governance.users_usage